# 12 — Support Mechanisms: CfD, PPA, FIT and Market Premium compared to naive bidding

In this notebook we compare five identical solar units that differ only in how they monetize
their generation:

1. **CfD** (Contract for Difference) — receives the difference between a fixed strike price and the market price for its contracted generation (two-sided: it pays back when the market price is above the strike).
2. **PPA** (Power Purchase Agreement) — sells its generation at a fixed price to a buyer.
3. **FIT** (Feed-in Tariff) — receives a fixed payment per MWh generated.
4. **Market Premium** — receives the difference between the strike price and the market price, but only when the market price is below the strike (one-sided).
5. **Naive bidding** — no support; bids its marginal cost (~0 €/MWh for solar) into the energy-only market and takes the full price risk.

All units sell their energy on the hourly day-ahead market (**EOM**). The four supported units additionally
bid into a **contract market** (`PayAsBidContractRole`), where the demand side acts as the buyer of the
support contracts (representing the state or a corporate buyer). Contract payments are settled
retrospectively based on the actual generation and the realized EOM price.

The rest of the system is deliberately simple:

- **1 gas power plant** with enough capacity to cover the whole demand at any time (the price-setting unit),
- **1 inelastic demand**.

The simulation runs for the **full year 2019** (hourly resolution). The evaluation at the end of the
notebook compares one week per season: winter, spring, summer and autumn.

In [ ]:
import importlib.util

# Check whether notebook is run in google colab
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    %pip install assume-framework

## 1. Imports

In [ ]:
import os
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import pandas as pd
from dateutil import rrule as rr
from dateutil import relativedelta as rd

from assume import World
from assume.common.forecaster import DemandForecaster, PowerplantForecaster
from assume.common.market_objects import MarketConfig, MarketProduct
from assume.markets.clearing_algorithms.contracts import PayAsBidContractRole
from assume.strategies.extended import SupportStrategy

os.makedirs("local_db", exist_ok=True)

db_uri = "sqlite:///local_db/assume_db.db"

## 2. Load input data

We use two data sources from the existing examples:

- **Demand** from `example_01a` (`demand_df.csv`) — an inelastic demand profile for the full year 2019.
- **Solar availability** from `example_03` (`availability_df.csv`) — the solar capacity factor time series for 2019.

Both files are given in 15-minute resolution and are resampled to the hourly resolution of the simulation.

In [ ]:
# paths to the example data (relative to the examples/notebooks folder)
input_01a = "../inputs/example_01a"
input_03 = "../inputs/example_03"

# load demand and availability (15 min resolution) and resample to hourly
demand_df = pd.read_csv(f"{input_01a}/demand_df.csv", index_col=0, parse_dates=True)
availability_df = pd.read_csv(f"{input_03}/availability_df.csv", index_col=0, parse_dates=True)

demand_df = demand_df.resample("1h").mean()
availability_df = availability_df.resample("1h").mean()

solar_availability = availability_df["Solar"]

print(f"Demand: min {demand_df['demand_EOM'].min():.0f} MW, max {demand_df['demand_EOM'].max():.0f} MW")
print(f"Solar availability: min {solar_availability.min():.2f}, max {solar_availability.max():.2f}")

### Quick look at the input data

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))

ax1.plot(demand_df.index, demand_df["demand_EOM"], color="tab:blue", label="Demand (MW)")
ax1.set_ylabel("Demand (MW)")
ax1.set_xlabel("")

ax2 = ax1.twinx()
ax2.plot(solar_availability.index, solar_availability, color="tab:orange", alpha=0.5, label="Solar availability")
ax2.set_ylabel("Solar availability (p.u.)")

fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.95))
fig.suptitle("Demand and solar availability in 2019")
fig.tight_layout()

## 3. Simulation setup

The simulation covers the full year 2019 with hourly time steps. We use a local sqlite database so no
docker infrastructure is required.

In [ ]:
start = datetime(2019, 1, 1)
end = datetime(2019, 12, 31)
simulation_id = "support_mechanisms"

index = pd.date_range(start=start, end=end, freq="h")

world = World(database_uri=db_uri, export_csv_path="outputs")

world.setup(
    start=start,
    end=end,
    save_frequency_hours=48,
    simulation_id=simulation_id,
    index=index,
)

## 4. Market design

Two markets are used:

### Energy-Only Market (EOM)
A standard hourly day-ahead auction with `pay_as_clear` clearing. All units (solar and gas) sell their
energy here; the demand buys here.

### Contract market ("Support")
A market with the `pay_as_bid_contract` clearing mechanism. Instead of energy delivery, **support
policies** are auctioned here. Units bid for a contract of a specific type (`CFD`, `PPA`, `FIT`,
`market_premium`) with a strike price (`contract_value`). The demand side bids as the buyer of these
contracts. Once a contract is cleared, the market role schedules recurring tasks which settle the
contract retrospectively — based on the actual generation of the selling unit and the realized EOM price.

The contract market opens **monthly** and each contract covers one month, but the cashflow is evaluated
**weekly** (see `evaluation_frequency`).

In [ ]:
# the support strategy is not registered by default and must be added manually
world.bidding_strategies["support"] = SupportStrategy

marketdesign = [
    MarketConfig(
        market_id="EOM",
        opening_hours=rr.rrule(rr.HOURLY, dtstart=start+timedelta(hours=2), until=end),
        opening_duration=timedelta(hours=1),
        market_mechanism="pay_as_clear",
        market_products=[MarketProduct(timedelta(hours=1), 1, timedelta(hours=1))],
        product_type="energy",
        volume_unit="MWh",
        price_unit="EUR/MWh",
        maximum_bid_price=3000,
        minimum_bid_price=-500,
        maximum_bid_volume=100000,
    ),
    MarketConfig(
        market_id="Support",
        opening_hours=rr.rrule(rr.MONTHLY, dtstart=start, until=end),
        opening_duration=timedelta(hours=1),
        market_mechanism="pay_as_bid_contract",
        market_products=[MarketProduct(rd.relativedelta(months=1), 1, timedelta(hours=1))],
        product_type="financial_support",
        additional_fields=[
            "sender_id",
            "contract",
            "eligible_lambda",
            "evaluation_frequency",
        ],
        supports_get_unmatched=True,
        param_dict={"allowed_contracts": ["CFD", "PPA", "FIT", "MPFIX"]},
        volume_unit="MW",
        price_unit="EUR/MWh",
        maximum_bid_price=3000,
        maximum_bid_volume=100000,
    ),
]

mo_id = "market_operator"
world.add_market_operator(id=mo_id)
for market_config in marketdesign:
    world.add_market(market_operator_id=mo_id, market_config=market_config)

## 5. Units

The units are grouped into three unit operators:

- **gas_operator** — the gas power plant. It is the price-setting unit and must be able to cover the
  whole demand at any time.
- **solar_operator** — the five solar units (1000 MW each). They all use the same availability time
  series, but differ in how they monetize their generation.
- **demand_operator** — the inelastic demand. It additionally acts as the **buyer of the support
  contracts** (representing the state or a corporate buyer paying the support).

> **Note:** the gas plant is placed in its own unit operator because the default eligibility check for
> support contracts (`is_co2emissionless`) requires *all* units of an operator to be emissionless.

The four supported solar units bid into the EOM *and* into the contract market with the `support`
strategy, each for exactly one contract type at a strike price of 80 €/MWh:

| Unit | Contract type | EOM strategy |
|---|---|---|
| `solar_cfd` | `CFD` | naive (marginal cost) |
| `solar_ppa` | `PPA` | naive (marginal cost) |
| `solar_fit` | `FIT` | naive (marginal cost) |
| `solar_mp` | `MPFIX` (market premium) | naive (marginal cost) |
| `solar_naive` | — | naive (marginal cost) |

The contract cashflow is evaluated **weekly** (`evaluation_frequency`), based on the actual generation
of the unit and the realized EOM price.

In [ ]:
world.add_unit_operator("gas_operator")
world.add_unit_operator("solar_operator")
world.add_unit_operator("demand_operator")

# --- gas plant: covers the whole demand at any time and sets the price ---
gas_forecaster = PowerplantForecaster(
    index,
    availability=1,
    fuel_prices={"natural gas": 25, "co2": 25},
)

world.add_unit(
    id="gas_plant",
    unit_type="power_plant",
    unit_operator_id="gas_operator",
    unit_params={
        "min_power": 0,
        "max_power": 4000,
        "efficiency": 0.6,
        "fuel_type": "natural gas",
        "emission_factor": 0.2,
        "additional_cost": 3.5,
        "technology": "gas",
        "bidding_strategies": {"EOM": "powerplant_energy_naive"},
    },
    forecaster=gas_forecaster,
)

# --- inelastic demand, also acting as buyer of the support contracts ---
demand_forecaster = DemandForecaster(index, demand=-demand_df["demand_EOM"])

world.add_unit(
    id="demand_unit",
    unit_type="demand",
    unit_operator_id="demand_operator",
    unit_params={
        "min_power": 0,
        # max_power is used as contract volume by the SupportStrategy
        "max_power": -4000,
        "technology": "demand",
        "bidding_strategies": {
            "EOM": "demand_energy_naive",
            "Support": "support",
},
        "bidding_params": {
            "contract_types": ["CFD", "PPA", "FIT", "MPFIX"],
            "contract_amount_fraction": 1.0,
            "evaluation_frequency": rr.WEEKLY,
        },
    },
    forecaster=demand_forecaster,
)

### The five solar units

All solar units share the same capacity (1000 MW), the same availability time series from example_03
and therefore the same generation profile — only their revenue scheme differs. Their marginal cost is
~0 €/MWh, so on the EOM they all bid at price 0.

In [ ]:
solar_forecaster = PowerplantForecaster(
    index,
    availability=solar_availability,
    fuel_prices={},
)

solar_units = {
    "solar_cfd": ["CFD"],
    "solar_ppa": ["PPA"],
    "solar_fit": ["FIT"],
    "solar_mp": ["MPFIX"],
    # the naive unit bids only on the EOM and takes the full price risk
    "solar_naive": [],
}

CONTRACT_VALUE = 80  # strike price of the support contracts in EUR/MWh

for unit_id, contract_types in solar_units.items():
    bidding_strategies = {"EOM": "powerplant_energy_naive"}
    bidding_params = {}
    if contract_types:
        bidding_strategies["Support"] = "support"
        bidding_params = {
            "contract_types": contract_types,
            "contract_value": CONTRACT_VALUE,
            "contract_amount_fraction": 1.0,
            "evaluation_frequency": rr.WEEKLY,
        }

    world.add_unit(
        id=unit_id,
        unit_type="power_plant",
        unit_operator_id="solar_operator",
        unit_params={
            "min_power": 0,
            "max_power": 1000,
            "efficiency": 1,
            "fuel_type": "renewable",
            "emission_factor": 0,
            "additional_cost": 0,
            "technology": "solar",
            "bidding_strategies": bidding_strategies,
            "bidding_params": bidding_params,
        },
        forecaster=solar_forecaster,
    )

## 6. Run the simulation

During the simulation:

- the **EOM** clears every hour (`pay_as_clear`) — solar bids at ~0 €/MWh, the gas plant at its
  marginal cost (~53.5 €/MWh), the demand bids its forecast at 3000 €/MWh,
- the **contract market** opens monthly and auctions one-month contracts of each type
  (`CFD`, `PPA`, `FIT`, `MPFIX`) between the solar units and the demand,
- for every cleared contract, the market role schedules **weekly settlement tasks** which pay the
  difference between the strike price and the realized EOM price (CfD, MPFIX), the fixed price for
  the actual generation (PPA, FIT) — based on the actual generation of the unit.

The full year takes a few minutes to simulate.

In [ ]:
world.run()

## 7. Verify the market results

Before evaluating individual weeks, we check that both markets worked as intended:

1. every monthly contract auction cleared for all four contract types,
2. the retrospective contract payments were booked to the units,
3. the EOM cleared in every hour with prices between 0 and 53.50 €/MWh (the gas plant price).

In [ ]:
import sqlite3

# sqlite3.connect does not accept URI schemes - strip the prefix
db_path = db_uri.replace("sqlite:///", "")
con = sqlite3.connect(db_path)

# --- contract market: all monthly auctions cleared ---
support_meta = pd.read_sql(
    f"SELECT * FROM market_meta WHERE market_id='Support' AND simulation='{simulation_id}'",
    con,
    parse_dates=["time"],
)
# the contract for the last month is not auctioned, as it would extend
# beyond the simulation end (product duration + first delivery)
print(f"Contract market clearings: {len(support_meta)} (expected: 4 contracts x 11 months = 44)")
print(f"Contracted volume per clearing: {support_meta['supply_volume'].unique()} MW")
print(f"Contract prices: {support_meta['price'].unique()} EUR/MWh")

# --- EOM: cleared every hour ---
eom_meta = pd.read_sql(
    f"SELECT * FROM market_meta WHERE market_id='EOM' AND simulation='{simulation_id}'",
    con,
    parse_dates=["time"],
)
print(f"\nEOM clearings: {len(eom_meta)} (expected: 8760)")
print(f"EOM price range: {eom_meta['price'].min():.2f} - {eom_meta['price'].max():.2f} EUR/MWh")

# --- cashflows per unit ---
unit_dispatch = pd.read_sql(
    f"SELECT * FROM unit_dispatch WHERE simulation='{simulation_id}'",
    con,
    parse_dates=["time"],
)
cashflows = unit_dispatch.groupby("unit")[
    ["energy_cashflow", "financial_support_cashflow"]
].sum()
cashflows["total"] = cashflows.sum(axis=1)
print("\nTotal cashflows over the year (EUR):")
print(cashflows.round(0))
con.close()

### Expected results

- All five solar units have **identical `energy_cashflow`** — they all sell the same generation on
  the EOM at the same price.
- `solar_cfd` and `solar_mp` receive **almost identical support payments**: the EOM price never exceeds
  the gas plant's marginal cost (~53.5 €/MWh), which is below the 80 €/MWh strike — so the one-sided
  market premium (`MPFIX`) pays the same as the two-sided CfD.
- `solar_fit` and `solar_ppa` receive 80 €/MWh for every MWh actually generated.
- `solar_naive` receives no support payments — its revenue is exactly the (volatile) EOM revenue.
- The demand unit pays the sum of all support payments.

## 8. Seasonal evaluation

We compare one week per season to see how the different support mechanisms perform under
different conditions:

- **Winter** — low solar availability, high demand, the gas plant sets the price in every hour,
- **Spring** — moderate solar availability, solar surplus pushes the EOM price to zero around midday,
- **Summer** — high solar availability, several hours with zero EOM prices at midday,
- **Autumn** — decreasing solar availability, the gas plant sets the price again.

Each figure shows:

1. the **solar availability** and the **EOM price**,
2. the **accepted bids on the EOM** for every unit (the five solar units have identical bids, as
   they all bid their marginal cost of ~0 €/MWh with the same availability),
3. the **cumulative cashflow** of the solar units — the steps are the weekly contract settlements,
4. the **weekly totals** of spot revenue and support payments per unit.

In [ ]:
# load the simulation results from the local database
# sqlite3.connect does not accept URI schemes - strip the prefix
db_path = db_uri.replace("sqlite:///", "")
con = sqlite3.connect(db_path)

unit_dispatch = pd.read_sql(
    f"SELECT * FROM unit_dispatch WHERE simulation='{simulation_id}'",
    con,
    parse_dates=["time"],
)

eom_meta = pd.read_sql(
    f"SELECT * FROM market_meta WHERE market_id='EOM' AND simulation='{simulation_id}'",
    con,
    parse_dates=["time"],
).set_index("time")

con.close()

# one dataframe per unit, indexed by time
dispatch = {
    unit: group.set_index("time").sort_index()
    for unit, group in unit_dispatch.groupby("unit")
}

solar_units = ["solar_cfd", "solar_ppa", "solar_fit", "solar_mp", "solar_naive"]
unit_colors = {
    "solar_cfd": "tab:red",
    "solar_ppa": "tab:purple",
    "solar_fit": "tab:green",
    "solar_mp": "tab:orange",
    "solar_naive": "tab:gray",
    "gas_plant": "tab:brown",
    "demand_unit": "black",
}

In [ ]:
solar_units[0]

In [ ]:
def plot_week(season: str, week_start: str):
    """Plot availability, accepted bids and cashflows for one week."""
    start_week = pd.Timestamp(week_start)
    end_week = start_week + pd.Timedelta(days=7)

    week_slice = slice(start_week, end_week - pd.Timedelta(hours=1))
    week_price = eom_meta["price"].loc[week_slice]
    week_availability = solar_availability.loc[week_slice]

    fig, axs = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(f"{season} — week of {week_start}", fontsize=14)

    # 1. solar availability and EOM price
    ax = axs[0, 0]
    ax.plot(week_availability.index, week_availability.values, color="tab:orange", label="solar availability")
    ax.set_ylabel("availability (p.u.)")
    ax.set_ylim(0, 1)
    ax2 = ax.twinx()
    ax2.plot(week_price.index, week_price.values, color="tab:blue", label="EOM price")
    ax2.set_ylabel("EOM price (EUR/MWh)")
    ax.set_title("Availability and EOM price")
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

    # 2. accepted bids on the EOM
    ax = axs[0, 1]
    for unit in solar_units + ["gas_plant"]:
        ax.plot(
            dispatch[unit].loc[week_slice, "power"],
            color=unit_colors[unit],
            alpha=0.9 if unit == "gas_plant" else 0.6,
            label=unit,
        )
    ax.set_title("Accepted bids on the EOM")
    ax.set_ylabel("power (MW)")
    ax.legend(loc="upper left", fontsize=8)

    # 3. cumulative cashflow of the solar units
    ax = axs[1, 0]
    for unit in solar_units:
        cashflow = dispatch[unit].loc[week_slice, ["energy_cashflow", "financial_support_cashflow"]].sum(axis=1)
        ax.plot(cashflow.index, cashflow.cumsum(), color=unit_colors[unit], label=unit)
    ax.set_title("Cumulative total cashflow")
    ax.set_ylabel("cashflow (EUR)")
    ax.legend(loc="upper left", fontsize=8)

    # 4. weekly totals per unit
    ax = axs[1, 1]
    weekly = pd.DataFrame(
        {
            unit: dispatch[unit]
            .loc[week_slice, ["energy_cashflow", "financial_support_cashflow"]]
            .sum()
            for unit in solar_units
        }
    ).T
    # this is a workaround to correctly show cashflow from financial support and from the EOM
    # for FIT and PPA contracts, the product should in principle be "energy" because these units are not allowed to sell their energy again on the EOM.
    # but then the contract payment is recorded as energy_cashflow, while financial_support_cashflow is 0.
    # if we configure FIT/PPA as financial_support, this no longer enforces the “energy cannot be sold elsewhere” behavior; the unit may also sell the same generation on the EOM.
    # in that case it earns financial_support_cashflow from the contract and energy_cashflow from the EOM, which is not what we want to visualize here.
    # for the purpose of this notebook, we just want to visualize the cashflows correctly, until a fix is applied
    # see https://github.com/assume-framework/assume/issues/926
    weekly.loc[["solar_ppa", "solar_fit"], "energy_cashflow"] = 0

    x = range(len(weekly))
    ax.bar(x, weekly["energy_cashflow"], width=0.4, label="spot revenue (EOM)")
    ax.bar(x, weekly["financial_support_cashflow"], width=0.4, bottom=weekly["energy_cashflow"], label="support payments")
    ax.set_ylim(top=weekly.sum(axis=1).max() * 1.1)
    ax.set_xticks(list(x))
    ax.set_xticklabels(weekly.index, rotation=45, ha="right", fontsize=8)
    ax.set_title("Weekly totals")
    ax.set_ylabel("cashflow (EUR)")
    ax.legend(fontsize=8)

    fig.tight_layout()
    plt.show()


plot_week("Winter", "2019-01-21")

In [ ]:
plot_week("Spring", "2019-04-15")

In [ ]:
plot_week("Summer", "2019-06-24")

In [ ]:
plot_week("Autumn", "2019-10-14")

## 9. Discussion

- **Spot revenue is identical** for all five solar units in every week — they have the same
  generation and all bid at ~0 €/MWh on the EOM.
- **CfD and market premium** (`MPFIX`) pay `(strike − EOM price) × generation` retrospectively.
  As the EOM price never exceeds the gas plant's marginal cost (~53.5 €/MWh), both pay the same
  amount — the difference between the two mechanisms only becomes visible when the market price
  rises above the strike price, in which case the CfD would claw back revenue while the market
  premium would not.
- **PPA and FIT** pay the fixed strike price for every MWh generated — the highest and most stable
  revenue in this scenario.
- **Naive bidding** takes the full price risk: in the spring and summer weeks the EOM price drops to
  zero around midday, so the spot revenue is significantly lower than for the supported units.
- The steps in the cumulative cashflow are the **weekly contract settlements** — the payments for
  the whole week are booked in a single hour, retrospectively.